## Samplesheet setup

In [1]:
import re
import pandas as pd
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from pathlib import Path

# =========================
# EXPERIMENT METADATA
# =========================

BASE_PATH = Path("/data/CARDPB2/iNDI/Production/AbPanel1")
NS = {'h': '43B2A954-E3C3-47E1-B392-6635266B0DD3/HarmonyV7'}

pseudocolor_map = {
    "DAPI":       "blue",
    "Brightfield": "gray",
    "Alexa 488":  "green",
    "Alexa 568":  "red",
    "Alexa 647":  "magenta",
}

mpl_colormaps = {
    "blue":    LinearSegmentedColormap.from_list("black_blue",    [(0,0,0), (0,0,1)]),
    "green":   LinearSegmentedColormap.from_list("black_green",   [(0,0,0), (0,1,0)]),
    "red":     LinearSegmentedColormap.from_list("black_red",     [(0,0,0), (1,0,0)]),
    "magenta": LinearSegmentedColormap.from_list("black_magenta", [(0,0,0), (1,0,1)]),
    "gray":    LinearSegmentedColormap.from_list("black_gray",    [(0,0,0), (1,1,1)]),
}

antigens = {
    "GM130":  "Golgi",
    "LAMP1":  "Lysosome",
    "G3BP1":  "Stress granule",
    "EEA1":   "Endosome",
    "TOMM20": "Mitochondria",
    "TUBB3":  "Microtubules",
    "TGN46":  "Golgi",
    "MAP2":   "Microtubules",
    "TUJ1":   "Microtubules",
    "DAPI":   "Nuclei",
}

experimental_design = pd.DataFrame({
    "Panel":      [1,       2],
    "DAPI":       ["DAPI",  "DAPI"],
    "Alexa 488":  ["TOMM20","RAB11A"],
    "Alexa 568":  ["EEA1",  "GM130"],
    "Alexa 647":  ["LAMP1", "TUJ1"],
}).melt(
    id_vars=["Panel"],
    value_vars=["DAPI", "Alexa 488", "Alexa 568", "Alexa 647"],
    var_name="Channel_name",
    value_name="Stain",
)

# Map stain → structure
antigen_keys = list(antigens.keys())
pattern = r"\b(" + "|".join(re.escape(k) for k in antigen_keys) + r")\b"
matched = experimental_design["Stain"].str.extract(pattern, flags=re.IGNORECASE)[0]
experimental_design["Structure"] = (
    matched.str.upper()
    .map({k.upper(): v for k, v in antigens.items()})
    .fillna("Unknown")
)


# =========================
# SAMPLESHEET BUILDER
# =========================

def _parse_filename(name):
    match = re.match(r"r(\d+)c(\d+)f(\d+)p(\d+)-ch(\d+)t(\d+)", name)
    return [int(g) for g in match.groups()] if match else [None] * 6


def build_samplesheet(measurement_id: str, panel: int = 1) -> pd.DataFrame:
    """
    Build a samplesheet for one measurement (plate).

    Parameters
    ----------
    measurement_id : str
        The experiment UUID, e.g. '028ebee9-afaf-4ff8-b435-af11714285dc'
    panel : int
        Which panel to join against in experimental_design (default 1).

    Returns
    -------
    pd.DataFrame
        Full samplesheet with filepaths, metadata, stain, and structure columns.
    """
    exp_path = BASE_PATH / measurement_id
    img_dir  = exp_path / "images"

    # --- parse experiment XML ---
    exp_xml   = next(exp_path.glob("*.xml"), None)
    index_xml = next((exp_path / "index").glob("*.xml"), None)

    exp_root   = ET.parse(exp_xml).getroot()
    index_root = ET.parse(index_xml).getroot()

    meas_id   = exp_root.find('h:MeasurementID', NS).text
    date      = exp_root.find('h:Date', NS).text
    plate_id  = index_root.find('.//h:PlateID', NS).text
    x_res     = float(index_root.find('.//h:ImageResolutionX', NS).text) * 1e6
    y_res     = float(index_root.find('.//h:ImageResolutionY', NS).text) * 1e6

    # --- parse channel info ---
    channels = []
    for map_el in index_root.findall(".//h:Map", NS):
        first = map_el.find("h:Entry", NS)
        if first is not None and first.find("h:ChannelName", NS) is not None:
            for entry in map_el.findall("h:Entry", NS):
                ch_id = entry.attrib.get("ChannelID")
                channels.append({
                    "ChannelID":    int(ch_id) if ch_id else None,
                    "Channel_name": entry.find("h:ChannelName", NS).text,
                    "Type":         entry.findtext("h:ChannelType", default=None, namespaces=NS),
                    "Excitation_nm": entry.findtext("h:MainExcitationWavelength", default=None, namespaces=NS),
                    "Emission_nm":   entry.findtext("h:MainEmissionWavelength", default=None, namespaces=NS),
                })
            break

    channel_df = pd.DataFrame(channels).sort_values("ChannelID").reset_index(drop=True)
    channel_df["Pseudocolor"]    = channel_df["Channel_name"].map(pseudocolor_map).fillna("gray")
    channel_df["MPL_colormap"]   = channel_df["Pseudocolor"].str.lower().map(mpl_colormaps)
    channel_df["Measurement_ID"] = meas_id
    channel_df["Measurement_date"] = date
    channel_df["Plate_ID"]       = plate_id
    channel_df["res_x"]          = x_res
    channel_df["res_y"]          = y_res

    # --- collect tiff files ---
    files = sorted(f for f in img_dir.rglob("*") if f.suffix.lower() == ".tiff")
    file_df = pd.DataFrame({
        "filepath":    files,
        "filename":    [f.name for f in files],
        "subdirectory": [str(f.parent.relative_to(img_dir)) for f in files],
    })
    file_df[["Row", "Column", "Frame", "Plane", "ChannelID", "Time"]] = (
        file_df["filename"].apply(lambda x: pd.Series(_parse_filename(x)))
    )

    # --- merge channel metadata ---
    merged = pd.merge(file_df, channel_df, on="ChannelID")
    merged["Panel"] = panel

    # --- merge experimental design (stain + structure) ---
    design = experimental_design[experimental_design["Panel"] == panel]
    samplesheet = pd.merge(merged, design, on=["Channel_name", "Panel"])

    return samplesheet

In [2]:
# =========================
# BUILD ALL SAMPLESHEETS
# =========================

# interested in plate 43 - phenotypes look a bit more on the 'extreme' side of things
# also consider plate 2

# 43: 4405a3b2-6b88-49b1-91f3-992e09ccbd16
#  2: 2b1c31ad-579d-4ea6-a782-251ea083ed6a

MEASUREMENT_IDS = [
    '4405a3b2-6b88-49b1-91f3-992e09ccbd16'
    ]

# MEASUREMENT_IDS = [p.name for p in BASE_PATH.iterdir() if p.is_dir()]
all_samplesheets = {mid: build_samplesheet(mid) for mid in MEASUREMENT_IDS}

# Access one from all loaded:
# samplesheet = all_samplesheets["028ebee9-afaf-4ff8-b435-af11714285dc"]

# Or concatenate all into one big samplesheet:
samplesheet_all = pd.concat(all_samplesheets.values(), ignore_index=True)

In [3]:
print(MEASUREMENT_IDS)

['4405a3b2-6b88-49b1-91f3-992e09ccbd16']


In [4]:
print(samplesheet_all)

                                                filepath  \
0      /data/CARDPB2/iNDI/Production/AbPanel1/4405a3b...   
1      /data/CARDPB2/iNDI/Production/AbPanel1/4405a3b...   
2      /data/CARDPB2/iNDI/Production/AbPanel1/4405a3b...   
3      /data/CARDPB2/iNDI/Production/AbPanel1/4405a3b...   
4      /data/CARDPB2/iNDI/Production/AbPanel1/4405a3b...   
...                                                  ...   
49275  /data/CARDPB2/iNDI/Production/AbPanel1/4405a3b...   
49276  /data/CARDPB2/iNDI/Production/AbPanel1/4405a3b...   
49277  /data/CARDPB2/iNDI/Production/AbPanel1/4405a3b...   
49278  /data/CARDPB2/iNDI/Production/AbPanel1/4405a3b...   
49279  /data/CARDPB2/iNDI/Production/AbPanel1/4405a3b...   

                        filename subdirectory  Row  Column  Frame  Plane  \
0      r01c03f01p01-ch01t01.tiff       r01c03    1       3      1      1   
1      r01c03f01p01-ch02t01.tiff       r01c03    1       3      1      1   
2      r01c03f01p01-ch03t01.tiff       r01c03    1 

## Find nuclei features

In [5]:
import pandas as pd
from pathlib import Path

NUCLEI_FEATURES_DIR = Path('/data/CARDPB2/iNDI/Production/nucleus_segmentation_result')

# Load all nuclei features files, keyed by measurement ID
all_nuclei_features = {}
for f in NUCLEI_FEATURES_DIR.iterdir():
    match = next((mid for mid in MEASUREMENT_IDS if mid in f.name), None)
    if match:
        nf = pd.read_csv(f)
        nf["experiment_name"] = match
        all_nuclei_features[match] = nf

nuclei_features = pd.concat(all_nuclei_features.values(), ignore_index=True)

In [6]:
print(nuclei_features.shape)
print(nuclei_features.dtypes)
print(nuclei_features.head())

(95365, 17)
label                   int64
area                  float64
mean_intensity        float64
max_intensity         float64
min_intensity         float64
std_intensity         float64
centroid-0            float64
centroid-1            float64
eccentricity          float64
solidity              float64
perimeter             float64
feret_diameter_max    float64
orientation           float64
major_axis_length     float64
minor_axis_length     float64
image_name             object
experiment_name        object
dtype: object
   label    area  mean_intensity  max_intensity  min_intensity  std_intensity  \
0      4  7994.0     1622.333500         3022.0          888.0     449.452798   
1      5  7060.0     1469.185836         2241.0          895.0     301.633720   
2      6  4758.0     1502.597940         2255.0          912.0     354.450561   
3      7  6021.0     1695.952500         3321.0          998.0     491.665024   
4      8  5937.0     1581.980798         4227.0          94

## Pipeline

In [7]:
import os
import re
import uuid
import numpy as np
import pandas as pd
import tifffile
import dask.array as da
import dask
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from skimage.measure import label, regionprops_table
from concurrent.futures import ProcessPoolExecutor, as_completed
from scipy.stats import norm
from scipy.ndimage import distance_transform_edt, gaussian_laplace, binary_fill_holes
from skimage import filters, morphology
from skimage.filters import threshold_triangle, threshold_otsu
from skimage.morphology import remove_small_objects, erosion, dilation
from skimage.morphology import disk as morph_disk

# =========================
# SEGMENTERS
# =========================

def process_golgi_mask(image, intensity_scaling_param=[9, 19], blur_sigma=1,
                       log_sigma=1.6, log_cutoff=0.02, low_thresh_minArea=1200,
                       minArea=10, thin_dist=1):
    m, s = norm.fit(image)
    stretch_min = max(m - intensity_scaling_param[0] * s, image.min())
    stretch_max = min(m + intensity_scaling_param[1] * s, image.max())
    image_norm = (np.clip(image, stretch_min, stretch_max) - stretch_min) / (stretch_max - stretch_min + 1e-12)
    blurred = filters.gaussian(image_norm, sigma=blur_sigma)
    thresh = threshold_triangle(blurred)
    img_low_thresh = remove_small_objects(blurred > thresh, min_size=low_thresh_minArea, connectivity=1)
    img_low_thresh = dilation(img_low_thresh, footprint=morph_disk(2))
    img_high_thresh = np.zeros_like(img_low_thresh)
    lab_low, num_obj = label(img_low_thresh, return_num=True, connectivity=1)
    for idx in range(num_obj):
        single_obj = lab_low == (idx + 1)
        local_otsu = threshold_otsu(blurred[single_obj > 0])
        img_high_thresh[np.logical_and(blurred > local_otsu * 0.98, single_obj)] = 1
    skeleton = morphology.medial_axis(img_high_thresh > 0)
    dist = distance_transform_edt(skeleton == 0)
    mask = dist > 1 + 1e-5
    thinned = np.logical_xor(img_high_thresh > 0, erosion(img_high_thresh > 0, morph_disk(thin_dist)))
    skele_mask = np.where(np.logical_and(mask, thinned), 0, img_high_thresh)
    log = -1 * (log_sigma**2) * gaussian_laplace(blurred, sigma=log_sigma)
    golgi_mask = remove_small_objects(np.logical_or(log > log_cutoff, skele_mask) > 0, min_size=minArea, connectivity=1)
    return binary_fill_holes(golgi_mask)


def process_lysosome_mask(image, intensity_scaling_param=[3, 19], blur_sigma=1,
                          log_params=((5.0, 0.09), (2.5, 0.07), (1.0, 0.01)),
                          vesselness_sigma=[1], vesselness_cutoff=0.15, min_area=15):
    m, s = norm.fit(image.ravel())
    stretch_min = max(m - intensity_scaling_param[0] * s, image.min())
    stretch_max = min(m + intensity_scaling_param[1] * s, image.max())
    image_norm = (np.clip(image, stretch_min, stretch_max) - stretch_min) / (stretch_max - stretch_min + 1e-12)
    blurred = filters.gaussian(image_norm, sigma=blur_sigma)
    log_mask = np.logical_or.reduce([(-1.0 * sig**2 * gaussian_laplace(blurred, sigma=sig)) > cut for sig, cut in log_params])
    vessel_mask = filters.frangi(blurred, sigmas=vesselness_sigma) > vesselness_cutoff
    return remove_small_objects(binary_fill_holes(np.logical_or(log_mask, vessel_mask)), min_size=min_area, connectivity=1)


def process_endosome_mask(image, intensity_scaling_param=[3, 19], blur_sigma=1.0,
                          log_params=((1.0, 0.03),), min_area=3):
    m, s = norm.fit(image.ravel())
    stretch_min = max(m - intensity_scaling_param[0] * s, image.min())
    stretch_max = min(m + intensity_scaling_param[1] * s, image.max())
    image_norm = (np.clip(image, stretch_min, stretch_max) - stretch_min) / (stretch_max - stretch_min + 1e-12)
    blurred = filters.gaussian(image_norm, sigma=blur_sigma)
    log_mask = np.logical_or.reduce([(-1.0 * sig**2 * gaussian_laplace(blurred, sigma=sig)) > cut for sig, cut in log_params])
    return remove_small_objects(binary_fill_holes(log_mask), min_size=min_area, connectivity=1)


def process_mitochondria_mask(image, intensity_scaling_param=[3.5, 15], blur_sigma=1.0,
                              log_params=((5.0, 0.09), (2.5, 0.07), (1.0, 0.01)),
                              vesselness_sigmas=(1.5,), vesselness_cutoff=0.16,
                              black_ridges=False, min_area=10, fill_holes=False):
    m, s = norm.fit(image.ravel())
    stretch_min = max(m - intensity_scaling_param[0] * s, image.min())
    stretch_max = min(m + intensity_scaling_param[1] * s, image.max())
    image_norm = (np.clip(image, stretch_min, stretch_max) - stretch_min) / (stretch_max - stretch_min + 1e-12)
    blurred = filters.gaussian(image_norm, sigma=blur_sigma)
    log_mask = np.logical_or.reduce([(-1.0 * sig**2 * gaussian_laplace(blurred, sigma=sig)) > cut for sig, cut in log_params])
    vessel_mask = filters.frangi(blurred, sigmas=vesselness_sigmas, black_ridges=black_ridges) > vesselness_cutoff
    combined = np.logical_or(log_mask, vessel_mask)
    if fill_holes:
        combined = binary_fill_holes(combined)
    return remove_small_objects(combined, min_size=min_area, connectivity=1)


# =========================
# CONFIG
# =========================

SEGMENTERS_BY_STRUCTURE = {
    "Golgi":        process_golgi_mask,
    "Lysosome":     process_lysosome_mask,
    "Endosome":     process_endosome_mask,
    "Mitochondria": process_mitochondria_mask,
}
ALLOWED_STRUCTURES = set(SEGMENTERS_BY_STRUCTURE.keys())

ROI_RADIUS  = 120
N_WORKERS   = int(os.environ.get("SLURM_CPUS_PER_TASK", os.cpu_count()))
SITE_KEYS   = ["subdirectory", "Row", "Column", "Frame", "Plane", "Time"]

# All skimage regionprops properties that work with intensity images.
# Remove any that are too slow or unsupported in your skimage version.
REGIONPROPS_PROPERTIES = [
    "label",
    "area",
    "area_bbox",
    "area_convex",
    "area_filled",
    "axis_major_length",
    "axis_minor_length",
    "bbox",
    "centroid",
    "centroid_local",
    "centroid_weighted",
    "centroid_weighted_local",
    "coords",                    # removed before saving (not serialisable)
    "eccentricity",
    "equivalent_diameter_area",
    "euler_number",
    "extent",
    "feret_diameter_max",
    "image",                     # removed before saving
    "image_convex",              # removed before saving
    "image_filled",              # removed before saving
    "image_intensity",           # removed before saving
    "inertia_tensor",
    "inertia_tensor_eigvals",
    "intensity_max",
    "intensity_mean",
    "intensity_min",
    "intensity_std",
    "moments",
    "moments_central",
    "moments_hu",
    "moments_normalized",
    "moments_weighted",
    "moments_weighted_central",
    "moments_weighted_hu",
    "moments_weighted_normalized",
    "num_pixels",
    "orientation",
    "perimeter",
    "perimeter_crofton",
    "slice",                     # removed before saving
    "solidity",
]

# Properties that can't be stored directly in a flat DataFrame — drop them.
_ARRAY_PROPS = {
    "coords", "image", "image_convex", "image_filled",
    "image_intensity", "slice",
}

# --- QC controls ---
SHOW_QC                 = True
QC_MAX_SITES            = 5
QC_MAX_TILES_PER_STRUCT = 8
QC_OUTPUT_DIR           = "./qc_figures"


# =========================
# HELPERS
# =========================

_site_re = re.compile(
    r"r(?P<Row>\d+)c(?P<Column>\d+)f(?P<Frame>\d+)p(?P<Plane>\d+)"
    r"-ch(?P<ChannelID>\d+)t(?P<Time>\d+)", re.IGNORECASE)

def parse_site_keys_from_filename(fname: str) -> dict:
    m = _site_re.search(str(fname))
    if not m:
        return {}
    d = m.groupdict()
    out = {k: d[k] for k in ("Row", "Column", "Frame", "Plane", "Time") if d.get(k)}
    out["subdirectory"] = f"r{out['Row'].zfill(2)}c{out['Column'].zfill(2)}"
    return out

def imread_plane(path, prefer_lazy=True, enforce_2d=True):
    with tifffile.TiffFile(path) as tf:
        page = tf.pages[0]
        shape = page.shape if page.ndim <= 2 else tuple(page.shape[1:])
        dtype = page.dtype
    if not prefer_lazy:
        arr = tifffile.imread(path)
        if enforce_2d and arr.ndim == 3 and arr.shape[0] == 1:
            arr = arr[0]
        return arr

    @dask.delayed
    def _read(path_):
        arr = tifffile.imread(path_)
        if arr.ndim == 3 and arr.shape[0] == 1:
            arr = arr[0]
        return arr

    return da.from_delayed(_read(path), shape=shape, dtype=dtype)

def bbox_from_center(y, x, r, H, W):
    yi, xi = int(round(y)), int(round(x))
    return max(yi - r, 0), min(yi + r + 1, H), max(xi - r, 0), min(xi + r + 1, W)

def get_row(site_df: pd.DataFrame, **filters):
    sel = site_df.copy()
    for k, v in filters.items():
        sel = sel[sel[k] == v]
    return None if sel.empty else sel.iloc[0]

def _draw_circles(ax, xs, ys, r):
    for y, x in zip(ys, xs):
        ax.add_patch(Circle((x, y), r, fill=False, edgecolor="red", linewidth=1))

def qc_show_three_panel(nuc_img, ch_img, global_seg, xs, ys, r, title_prefix):
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(nuc_img, cmap="gray"); axes[0].set_title(f"{title_prefix} – DAPI");    axes[0].axis("off")
    axes[1].imshow(ch_img,  cmap="gray"); axes[1].set_title(f"{title_prefix} – Channel"); axes[1].axis("off")
    axes[2].imshow(ch_img,  cmap="gray"); axes[2].imshow(global_seg, alpha=0.4)
    axes[2].set_title(f"{title_prefix} – Overlay"); axes[2].axis("off")
    for ax in axes:
        _draw_circles(ax, xs, ys, r)
    plt.tight_layout()
    plt.show()

def qc_tile_gallery(ch_img, tiles, seg_tiles, title_prefix, max_tiles=8):
    if max_tiles <= 0 or len(tiles) == 0:
        return
    n = min(max_tiles, len(tiles))
    fig, axes = plt.subplots(1, n, figsize=(3 * n, 3))
    if n == 1:
        axes = [axes]
    for ax, (tile, seg) in zip(axes, list(zip(tiles, seg_tiles))[:n]):
        ax.imshow(tile, cmap="gray"); ax.imshow(seg, alpha=0.4); ax.axis("off")
    fig.suptitle(f"{title_prefix} – example ROI tiles", y=1.02)
    plt.tight_layout()
    plt.show()

def _save_roi_mask(seg_tile, structure, dapi_fname, nucleus_id, out_dir):
    dapi_s = str(dapi_fname).replace("/", "_")
    fname  = f"{structure}__{dapi_s}__NucID={nucleus_id}.png"
    path   = os.path.join(out_dir, fname)
    plt.imsave(path, seg_tile.astype(np.uint8) * 255, cmap="gray", vmin=0, vmax=255)


def _write_site_parquet(summary_df, instance_df, base_dir, mid, dapi_name):
    os.makedirs(base_dir, exist_ok=True)
    uid    = uuid.uuid4().hex[:8]
    mid_s  = str(mid).replace("/", "_")
    dapi_s = str(dapi_name).replace("/", "_")
    summary_path  = os.path.join(base_dir, f"summary__MID={mid_s}__DAPI={dapi_s}__{uid}.parquet")
    instance_path = os.path.join(base_dir, f"instance__MID={mid_s}__DAPI={dapi_s}__{uid}.parquet")
    summary_df.to_parquet(summary_path,  index=False)
    instance_df.to_parquet(instance_path, index=False)
    return summary_path, instance_path


def _regionprops_for_tile(seg_tile, tile, y0, x0, props=REGIONPROPS_PROPERTIES):
    """
    Run regionprops_table on a single segmented tile and return a DataFrame.
    Coordinates in the returned table are global (offset by y0, x0).
    Non-serialisable array properties are dropped.
    """
    obj_labels = label(seg_tile, connectivity=1)
    if obj_labels.max() == 0:
        return pd.DataFrame()

    wanted = [p for p in props if p not in _ARRAY_PROPS]
    rpt = regionprops_table(obj_labels, intensity_image=tile, properties=wanted)
    df  = pd.DataFrame(rpt)

    # Shift centroid / bbox coordinates to global image space
    for col in df.columns:
        if col.startswith("centroid-0") or col.startswith("bbox-0") or col.startswith("bbox-2"):
            df[col] = df[col] + y0
        elif col.startswith("centroid-1") or col.startswith("bbox-1") or col.startswith("bbox-3"):
            df[col] = df[col] + x0
        elif col.startswith("centroid_local") or col.startswith("centroid_weighted_local"):
            pass  # keep tile-local; already meaningful as offsets

    return df


# =========================
# CORE: process_site
# Returns (summary_df, instance_df)
# =========================

def process_site(site_meta, site_df, nuclei_features, segmenters_by_structure,
                 r=ROI_RADIUS, do_qc=False, mask_output_dir=None):
    site_df = site_df.copy()
    for c in ("Stain", "Structure", "filename"):
        if c in site_df.columns:
            site_df[c] = site_df[c].astype(str).str.strip()
    site_df["Stain"] = site_df["Stain"].str.upper()

    dapi_row = get_row(site_df, Stain="DAPI")
    if dapi_row is None:
        return None, None

    dapi_fname = str(dapi_row["filename"]).strip()
    nuc_da     = imread_plane(dapi_row["filepath"])
    nuc_img    = np.asarray(nuc_da.compute())
    H, W       = nuc_img.shape[:2]

    nf = nuclei_features.copy()
    key_col = "filename"
    nf[key_col] = nf[key_col].astype(str).str.strip()

    props_fname = str(site_meta.get("props_filename", site_meta.get("dapi_filename", ""))).strip()
    nf_sel = nf[nf[key_col] == props_fname]
    if nf_sel.empty:
        nf_sel = nf[nf[key_col] == dapi_fname]
    if nf_sel.empty:
        return None, None

    ys     = nf_sel["centroid-0"].astype(float).to_numpy()
    xs     = nf_sel["centroid-1"].astype(float).to_numpy()
    labels = nf_sel["label"].astype(int).to_numpy() if "label" in nf_sel.columns else np.arange(1, len(xs) + 1)

    canon  = {k.lower(): k for k in segmenters_by_structure}
    site_df["Structure"] = site_df["Structure"].str.lower().map(canon).fillna(site_df["Structure"])
    todo   = [s for s in sorted(set(site_df["Structure"].unique()) & set(segmenters_by_structure))
              if callable(segmenters_by_structure.get(s))]
    if not todo:
        return None, None

    summary_rows, instance_frames = [], []

    # Common metadata shared by every row of this site
    _site_meta_cols = {
        "Measurement_ID": site_meta.get("Measurement_ID"),
        "DAPI_filename":  dapi_fname,
    }

    for structure in todo:
        seg_fn = segmenters_by_structure[structure]
        row    = site_df.loc[site_df["Structure"] == structure].iloc[0]
        ch_da  = imread_plane(row["filepath"])
        if ch_da.shape[:2] != (H, W):
            raise ValueError(f"Shape mismatch @ {structure}: {ch_da.shape} vs {(H, W)}")

        ch_img = np.asarray(ch_da.compute())

        global_seg                  = np.zeros((H, W), dtype=bool)
        example_tiles, example_segs = [], []

        _struct_meta = {
            **_site_meta_cols,
            "subdirectory":    row.get("subdirectory"),
            "Row":             row.get("Row"),
            "Column":          row.get("Column"),
            "Frame":           row.get("Frame"),
            "Plane":           row.get("Plane"),
            "Time":            row.get("Time"),
            "channel_filename": row["filename"],
            "Structure":       structure,
            "Stain":           row.get("Stain"),
        }

        for lab, y, x in zip(labels, ys, xs):
            y0, y1, x0, x1 = bbox_from_center(y, x, r, H, W)
            tile = ch_img[y0:y1, x0:x1]
            yy, xx = np.ogrid[y0:y1, x0:x1]
            roi      = (yy - y)**2 + (xx - x)**2 <= r**2
            seg_tile = seg_fn(np.where(roi, tile, 0)) & roi
            global_seg[y0:y1, x0:x1] |= seg_tile

            if mask_output_dir:
                os.makedirs(mask_output_dir, exist_ok=True)
                _save_roi_mask(seg_tile, structure, dapi_fname, int(lab), mask_output_dir)

            if do_qc and len(example_tiles) < QC_MAX_TILES_PER_STRUCT:
                example_tiles.append(tile)
                example_segs.append(seg_tile)

            # ── per-nucleus summary (unchanged from original) ──────────────
            obj_labels_arr = label(seg_tile, connectivity=1)
            count_obj  = int(obj_labels_arr.max())
            vals       = tile[seg_tile]
            area_px    = int(seg_tile.sum())
            roi_area   = int(roi.sum())
            coverage_frac = (area_px / roi_area) if roi_area else 0.0
            area_px_mean  = (area_px / count_obj) if count_obj else 0.0

            if vals.size:
                int_max    = float(vals.max())
                int_sum    = float(vals.sum())
                int_mean   = float(vals.mean())
                int_median = float(np.median(vals))
                int_std    = float(vals.std())
            else:
                int_max = int_sum = 0.0
                int_mean = int_median = int_std = np.nan

            int_cv = (int_std / int_mean) if (not np.isnan(int_mean) and int_mean != 0) else np.nan

            rad      = np.sqrt((yy - y)**2 + (xx - x)**2)
            rad_norm = np.zeros_like(rad, dtype=float)
            rad_norm[roi] = rad[roi] / float(r)
            rad_obj  = rad_norm[seg_tile]
            rad_n    = rad_obj.size

            if rad_n:
                radial_mean         = float(rad_obj.mean())
                radial_p25          = float(np.percentile(rad_obj, 25))
                radial_p50          = float(np.percentile(rad_obj, 50))
                radial_p75          = float(np.percentile(rad_obj, 75))
                inner_frac          = float((rad_obj < 0.33).sum()) / rad_n
                mid_frac            = float(((rad_obj >= 0.33) & (rad_obj < 0.66)).sum()) / rad_n
                outer_frac          = float((rad_obj >= 0.66).sum()) / rad_n
                boundary_touch_frac = float(((r - rad[seg_tile]) <= 5).sum()) / rad_n
            else:
                radial_mean = radial_p25 = radial_p50 = radial_p75 = np.nan
                inner_frac  = mid_frac = outer_frac = boundary_touch_frac = 0.0

            summary_rows.append({
                **_struct_meta,
                "Nucleus_ID": int(lab),
                "area_px": area_px,
                "coverage_frac": coverage_frac,
                "organelle_count": count_obj,
                "average_organelle_area": area_px_mean,
                "max_f_intensity": int_max,
                "sum_f_intensity": int_sum,
                "mean_f_intensity": int_mean,
                "median_f_intensity": int_median,
                "CoefOfVar_intensity": int_cv,
                "radial_mean": radial_mean,
                "radial_p25": radial_p25, "radial_p50": radial_p50, "radial_p75": radial_p75,
                "inner_frac": inner_frac, "mid_frac": mid_frac, "outer_frac": outer_frac,
                "boundary_touch_frac": boundary_touch_frac,
            })

            # ── per-instance regionprops ───────────────────────────────────
            inst_df = _regionprops_for_tile(seg_tile, tile, y0, x0)
            if not inst_df.empty:
                inst_df.insert(0, "Nucleus_ID", int(lab))
                for k, v in reversed(list(_struct_meta.items())):
                    inst_df.insert(0, k, v)
                instance_frames.append(inst_df)

        if do_qc:
            title = f"{structure} | {dapi_fname}"
            qc_show_three_panel(nuc_img, ch_img, global_seg, xs, ys, r, title_prefix=title)
            if QC_MAX_TILES_PER_STRUCT > 0:
                qc_tile_gallery(ch_img, example_tiles, example_segs,
                                title_prefix=title, max_tiles=QC_MAX_TILES_PER_STRUCT)

    summary_df  = pd.DataFrame(summary_rows) if summary_rows else pd.DataFrame()
    instance_df = pd.concat(instance_frames, ignore_index=True) if instance_frames else pd.DataFrame()

    return summary_df, instance_df


# =========================
# SITE ITERATION
# =========================

def iter_sites_single_panel(samplesheet, nuclei_features):
    ss = samplesheet.copy()
    ss["filename"] = ss["filename"].astype(str).str.strip()
    nf = nuclei_features.copy()
    nf["image_name"] = nf["image_name"].astype(str).str.strip()

    site_keys = ["Row", "Column", "Frame", "Plane", "Time"]

    for fname in nf["image_name"].unique():
        matched = ss.loc[ss["filename"] == fname]
        if matched.empty:
            keys = parse_site_keys_from_filename(fname)
            if not keys:
                continue
            mask = np.ones(len(ss), dtype=bool)
            for k, v in keys.items():
                if k in ss.columns:
                    mask &= ss[k].astype(str).str.strip() == str(v).strip()
            matched = ss.loc[mask]

        if matched.empty:
            continue

        row0 = matched.iloc[0]
        mask = np.ones(len(ss), dtype=bool)
        for k in site_keys:
            if k in ss.columns:
                mask &= ss[k].astype(str).str.strip() == str(row0[k]).strip()
        site_df = ss.loc[mask].copy()

        if site_df.empty:
            continue

        site_meta = {
            "Measurement_ID": site_df["Measurement_ID"].iloc[0] if "Measurement_ID" in site_df.columns else None,
            "dapi_filename": fname,
            "props_filename": fname,
        }
        yield site_meta, site_df


# =========================
# WORKER
# =========================

def _process_site_worker(args):
    site_meta, site_df, nuclei_features, r, do_qc, qc_output_dir, segmenters = args
    nuclei_features = nuclei_features.copy().rename(columns={"image_name": "filename"})

    if do_qc:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        os.makedirs(qc_output_dir, exist_ok=True)
        _orig_show = plt.show
        _site_id   = str(site_meta.get("dapi_filename", "site")).replace("/", "_")
        _fig_count = [0]

        def _save_instead(*a, **kw):
            fig  = plt.gcf()
            path = os.path.join(qc_output_dir, f"{_site_id}_fig{_fig_count[0]}.png")
            fig.savefig(path, bbox_inches="tight", dpi=100)
            plt.close(fig)
            _fig_count[0] += 1

        plt.show = _save_instead

    try:
        return process_site(site_meta, site_df, nuclei_features, segmenters, r=r, do_qc=do_qc,
                            mask_output_dir=qc_output_dir if do_qc else site_meta.get("mask_output_dir"))
    except Exception as e:
        print(f"[worker] {site_meta.get('dapi_filename')} failed: {e}")
        return None, None
    finally:
        if do_qc:
            plt.show = _orig_show


# =========================
# PARALLEL run_all
# Returns (summary_df, instance_df)
# =========================

def run_all_parallel(
    samplesheet, nuclei_features, segmenters_by_structure,
    r=ROI_RADIUS, n_workers=N_WORKERS, output_dir=None,
    show_qc=SHOW_QC, qc_max_sites=QC_MAX_SITES, qc_output_dir=QC_OUTPUT_DIR,
):
    samplesheet     = samplesheet.copy()
    nuclei_features = nuclei_features.copy()
    samplesheet["filename"]       = samplesheet["filename"].astype(str).str.strip()
    nuclei_features["image_name"] = nuclei_features["image_name"].astype(str).str.strip()

    all_sites = list(iter_sites_single_panel(samplesheet, nuclei_features))
    total     = len(all_sites)
    print(f"Found {total} sites — dispatching to {n_workers} workers")

    qc_count, args_list = 0, []
    for site_meta, site_df in all_sites:
        do_qc = show_qc and (qc_count < qc_max_sites)
        if do_qc:
            qc_count += 1

        # Subset nuclei_features to only this site's rows before pickling to worker
        fname   = site_meta["props_filename"]
        nf_site = nuclei_features[nuclei_features["image_name"] == fname]
        if nf_site.empty:
            keys = parse_site_keys_from_filename(fname)
            mask = np.ones(len(nuclei_features), dtype=bool)
            for k, v in keys.items():
                if k in nuclei_features.columns:
                    mask &= nuclei_features[k].astype(str).str.strip() == str(v).strip()
            nf_site = nuclei_features[mask]

        site_meta["mask_output_dir"] = output_dir  # workers write masks alongside parquets
        args_list.append((site_meta, site_df, nf_site, r, do_qc, qc_output_dir, segmenters_by_structure))

    summary_parts, instance_parts = [], []
    completed = 0

    with ProcessPoolExecutor(max_workers=n_workers) as pool:
        futures = {pool.submit(_process_site_worker, args): args[0] for args in args_list}
        for fut in as_completed(futures):
            site_meta  = futures[fut]
            completed += 1
            try:
                summary_df, instance_df = fut.result()
                has_summary  = summary_df  is not None and not summary_df.empty
                has_instance = instance_df is not None and not instance_df.empty

                if has_summary or has_instance:
                    if output_dir:
                        _write_site_parquet(
                            summary_df  if has_summary  else pd.DataFrame(),
                            instance_df if has_instance else pd.DataFrame(),
                            output_dir,
                            site_meta.get("Measurement_ID", "unknown"),
                            site_meta["dapi_filename"],
                        )
                    else:
                        if has_summary:
                            summary_parts.append(summary_df)
                        if has_instance:
                            instance_parts.append(instance_df)

                    n_inst = len(instance_df) if has_instance else 0
                    print(f"[{completed}/{total}] ✓ {site_meta['dapi_filename']} "
                          f"→ {len(summary_df) if has_summary else 0} summary rows, "
                          f"{n_inst} instance rows")
                else:
                    print(f"[{completed}/{total}] – {site_meta['dapi_filename']} → no data")

            except Exception as e:
                print(f"[{completed}/{total}] ✗ {site_meta['dapi_filename']} → {e}")

    print("\nAll sites processed.")

    if output_dir:
        s_parts = [pd.read_parquet(os.path.join(output_dir, f))
                   for f in os.listdir(output_dir) if f.startswith("summary__") and f.endswith(".parquet")]
        i_parts = [pd.read_parquet(os.path.join(output_dir, f))
                   for f in os.listdir(output_dir) if f.startswith("instance__") and f.endswith(".parquet")]
        summary_df  = pd.concat(s_parts, ignore_index=True) if s_parts else pd.DataFrame()
        instance_df = pd.concat(i_parts, ignore_index=True) if i_parts else pd.DataFrame()
    else:
        summary_df  = pd.concat(summary_parts,  ignore_index=True) if summary_parts  else pd.DataFrame()
        instance_df = pd.concat(instance_parts, ignore_index=True) if instance_parts else pd.DataFrame()

    return summary_df, instance_df

In [8]:
# import subprocess, os

# MEASUREMENT = "028ebee9-afaf-4ff8-b435-af11714285dc"
# SCRATCH = f"/lscratch/{os.environ['SLURM_JOB_ID']}/{MEASUREMENT}"
# SRC = f"/data/CARDPB2/iNDI/Production/{MEASUREMENT}/images"

# # Stage just one well to scratch
# WELL = "r02c11"
# os.makedirs(SCRATCH, exist_ok=True)
# subprocess.run(["rsync", "-a", f"{SRC}/{WELL}/", f"{SCRATCH}/{WELL}/"], check=True)

# # Point samplesheet to scratch for just this well
# test_ss = samplesheet[samplesheet["subdirectory"] == WELL].copy()
# test_ss["filepath"] = test_ss["filepath"].astype(str).str.replace(SRC, SCRATCH, regex=False)

# # Subset nuclei features to match
# nuclei_features = all_nuclei_features["028ebee9-afaf-4ff8-b435-af11714285dc"]
# test_nf = nuclei_features[nuclei_features["image_name"].str.contains(WELL, regex=False)]

# # Run on a few frames only
# test_nf = test_nf[test_nf["image_name"].str.contains("f01|f02|f03", regex=True)]

# # Run
# results = run_all_parallel(test_ss, test_nf, SEGMENTERS_BY_STRUCTURE, r=ROI_RADIUS)
# print(results.shape)
# results.head()

In [ ]:
import subprocess, os
import pandas as pd

SCRATCH_BASE = f"/lscratch/{os.environ['SLURM_JOB_ID']}"
SRC_BASE     = "/data/CARDPB2/iNDI/Production/AbPanel1"
OUTPUT_DIR   = "/data/CARDPB2/iNDI/JaneliaTest/organelle_features"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Cap workers per experiment to avoid OOM mid-job.
# Full cpu_count can spike memory when many large TIFFs are in-flight simultaneously.
N_WORKERS_PER_EXP = max(1, N_WORKERS // 2)

all_summary_dfs  = []
all_instance_dfs = []

for mid in MEASUREMENT_IDS:
    print(f"\n{'='*60}")
    print(f"Processing: {mid}")
    print(f"{'='*60}")

    if mid not in all_samplesheets or mid not in all_nuclei_features:
        print(f"  Skipping {mid} — missing samplesheet or nuclei features")
        continue

    ss = all_samplesheets[mid].copy()
    nf = all_nuclei_features[mid].copy()

    scratch_dir = f"{SCRATCH_BASE}/{mid}"
    src_dir     = f"{SRC_BASE}/{mid}/images"

    try:
        subprocess.run(
            ["rsync", "-a", "--info=progress2", f"{src_dir}/", f"{scratch_dir}/"],
            check=True
        )
    except subprocess.CalledProcessError as e:
        print(f"  rsync failed for {mid}: {e} — skipping")
        continue

    ss["filepath"] = ss["filepath"].astype(str).str.replace(src_dir, scratch_dir, regex=False)

    try:
        summary_df, instance_df = run_all_parallel(
            ss, nf, SEGMENTERS_BY_STRUCTURE,
            r=ROI_RADIUS,
            n_workers=N_WORKERS_PER_EXP,
            output_dir=OUTPUT_DIR,
        )
        print(f"  Done: {len(summary_df)} summary rows, {len(instance_df)} instance rows")
        if not summary_df.empty:
            all_summary_dfs.append(summary_df)
        if not instance_df.empty:
            all_instance_dfs.append(instance_df)

    except Exception as e:
        print(f"  run_all_parallel failed for {mid}: {e}")

    finally:
        # Always attempt cleanup; don't raise if scratch_dir never existed
        if os.path.exists(scratch_dir):
            result = subprocess.run(["rm", "-rf", scratch_dir])
            if result.returncode == 0:
                print(f"  Cleared scratch: {scratch_dir}")
            else:
                print(f"  Warning: scratch cleanup may have failed for {scratch_dir}")

# Concatenate everything
organelle_features = pd.concat(all_summary_dfs,  ignore_index=True) if all_summary_dfs  else pd.DataFrame()
instance_features  = pd.concat(all_instance_dfs, ignore_index=True) if all_instance_dfs else pd.DataFrame()

print(f"\nFinal summary shape:  {organelle_features.shape}")
print(f"Final instance shape: {instance_features.shape}")


Processing: 4405a3b2-6b88-49b1-91f3-992e09ccbd16
245,672,854,245 100%  217.19MB/s    0:17:58 (xfr#49284, to-chk=0/49637)   


Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7ffff4172cb0>>
Traceback (most recent call last):
  File "/data/kelpschdj/conda/envs/indi_project/lib/python3.10/site-packages/ipykernel/ipkernel.py", line 781, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 


In [ ]:
# OUTPUT_DIR = f"/data/CARDPB2/iNDI/Production/Organelle_segmentation/Panel_1/results/{MEASUREMENT}"
# /data/kelpschdj/iNDI/Production/Organelle_segmentation/Panel_1


# results = run_all_parallel(
#     samplesheet,
#     nuclei_features,
#     SEGMENTERS_BY_STRUCTURE,
#     r=ROI_RADIUS,
#     output_dir=OUTPUT_DIR
# )

In [ ]:
# import dask.dataframe as dd

# df = dd.read_parquet("/data/kelpschdj/iNDI/Production/Organelle_segmentation/Panel_1/organelle_features/*.parquet")
# combined = df.compute()

# print(combined.shape)
# print(combined.dtypes)

# combined.to_parquet(
#     "organelle_segmentation_AbPanel1.parquet",
#     engine="pyarrow",
#     index=False,
#     compression="zstd"
# )

(6178518, 29)
Measurement_ID            string[pyarrow]
subdirectory              string[pyarrow]
Row                                 int64
Column                              int64
Frame                               int64
Plane                               int64
Time                                int64
DAPI_filename             string[pyarrow]
channel_filename          string[pyarrow]
Structure                 string[pyarrow]
Stain                     string[pyarrow]
Nucleus_ID                          int64
area_px                             int64
coverage_frac                     float64
organelle_count                     int64
average_organelle_area            float64
max_f_intensity                   float64
sum_f_intensity                   float64
mean_f_intensity                  float64
median_f_intensity                float64
CoefOfVar_intensity               float64
radial_mean                       float64
radial_p25                        float64
radial_p50          

In [ ]:
# combined["Measurement_ID"].unique()
# 'd7e04b5c-a253-42ff-859f-2af0e6047a00' is not actually complete - this is where the job stalled out
# ^ reran, there may be some duplicate data? verify the number of organelle segmentations per nucleus accross each plate